# Figure S2D — 1-Year Cumulative Incidence by Age Group

Self-contained notebook (no `descriptive_config`/`prepare_data` imports). KM-based 1-year cumulative incidence per toxicity, stratified by age group at line-1 start,
with 95% CI error bars. Age groups are compared with **multivariable** Cox proportional-hazards
models (adjusted for sex and cancer type).

**Cohort:** first line of therapy only. `age_at_lot_start`, `line1_start`, and censoring are all
derived from the *first* LOT row per patient, selected by `idxmin` on `lot` (not
`.groupby().first()`, which takes the first non-null value per column independently and can mix
values across lines), matching S2A / S2B / S2C.

**Censoring (from `line1`):** `min(lot_end + 180d, next_lot_start, death, last_fu)`, measured from `line1_start`.

**Outputs:**
- `Age_group_prevalance_S2D.pdf` — the figure
- `Age_group_prevalance_S2D_cumulative_incidence.csv` — the underlying 1-year CI / 95% CI values per age group × toxicity

**Formatting (Nature compliance, consistent with the rest of Figure 2 / S2):**
- Arial only (no fallback substitution), `pdf.fonttype = 42` / `ps.fonttype = 42` (text stays editable, not outlined)
- Tiered text sizes: 7pt axis labels, 6pt tick labels, 5pt legend
- No `bbox_inches='tight'` on save — figure is placed in Illustrator at 100% scale, unresized


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
%matplotlib inline

matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from lifelines import KaplanMeierFitter

warnings.filterwarnings('ignore')

# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")


## Paths

Notebook lives in `figure 2/scripts/`. Data lives in the sibling `figure 2/data/` folder; outputs go to
`figure 2/results/supp/S2D_LLM_Age_Prevalence`


In [ ]:
NOTEBOOK_DIR = os.getcwd()
FIGURES_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
DATA_DIR = os.path.join(FIGURES_DIR, 'figures_data', 'figure 2', 'data')
RESULTS_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results', 'supp', 'S2D_LLM_Age_Prevalence'))
os.makedirs(RESULTS_DIR, exist_ok=True)

LLM_PATIENT_PATH = os.path.join(DATA_DIR, 'llm_calls_patient_level_84k.csv')
LLM_BATCH_PATH = os.path.join(DATA_DIR, 'llm_calls_batch_level_84k.csv')

PDF_OUT = os.path.join(RESULTS_DIR, 'Age_Group_Prevalance_S2D.pdf')
CSV_OUT = os.path.join(RESULTS_DIR, 'Age_Group_Prevalance_S2D_cumulative_incidence.csv')

for p in [LLM_PATIENT_PATH, LLM_BATCH_PATH]:
    print(('FOUND   ' if os.path.exists(p) else 'MISSING '), p)

## Constants

Toxicity list/display names and age-group colors, matching `descriptive_config.py` and `fig2i_age.py`.


In [ ]:
TOXICITY_COLUMNS = ['liver_toxicity', 'hypothyroidism', 'pneumonitis',
                    'colitis', 'adrenal_insufficiency', 'hyperthyroidism']

TOXICITY_DISPLAY = {
    'pneumonitis': 'Pneumonitis', 'adrenal_insufficiency': 'Adrenal Insufficiency',
    'liver_toxicity': 'Liver Toxicity', 'colitis': 'Colitis',
    'hyperthyroidism': 'Hyperthyroidism', 'hypothyroidism': 'Hypothyroidism',
}

AGE_GROUPS = ['<50', '50-65', '65-80', '>80']
AGE_GROUP_COLORS = {'<50': '#3498DB', '50-65': '#2ECC71', '65-80': '#E67E22', '>80': '#E74C3C'}

T_MONTHS = 12.0

import re

def standardize_mrn(mrn):
    if pd.isna(mrn):
        return None
    try:
        digits = re.findall(r'\d+', str(mrn).strip().strip("'\""))
        return str(int(digits[0])).zfill(8) if digits else None
    except (ValueError, TypeError):
        return None


## Build `line1` (first-line censoring) and `patient_covars` (first LOT per patient)

In [ ]:
COVARS_PATH = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026', 'llm84k_pneumonitis_grade0_20260630.csv')
covars = pd.read_csv(COVARS_PATH, low_memory=False)
covars['mrn'] = covars['mrn'].apply(standardize_mrn)
covars = covars[covars['mrn'].notna()].copy()

covars['lot'] = pd.to_numeric(covars['lot'], errors='coerce')
covars['age_at_lot_start'] = pd.to_numeric(covars['age_at_lot_start'], errors='coerce')
covars['lot_start'] = pd.to_datetime(covars['lot_start'], errors='coerce')
covars['censor_days'] = pd.to_numeric(covars['t_cutoff_lot'], errors='coerce')
covars = covars[covars['lot'].notna()].copy()
covars = covars.sort_values(['mrn', 'lot'])

# ---- line1: first LOT per patient, with pre-computed censoring ----
# idxmin selects the line-1 ROW. NOT .groupby().first(), which takes the first non-null value
# per column independently and can pair a line-1 start date with a later line's age/censoring.
# Same construction as S2C.
covars_valid = covars[covars['lot_start'].notna()].copy()
idx = covars_valid.sort_values(['mrn', 'lot']).groupby('mrn')['lot'].idxmin()
line1 = (covars_valid.loc[idx, ['mrn', 'lot', 'lot_start', 'censor_days']]
         .rename(columns={'lot': 'line1_lot', 'lot_start': 'line1_start'}))
line1 = line1[np.isfinite(line1['censor_days']) & (line1['censor_days'] > 0)].copy()
print(f'{len(line1):,} patients with line 1 dates and valid censoring')
print(f"  line-1 LOT value distribution: {line1['line1_lot'].value_counts().sort_index().to_dict()}")

# ---- patient_covars: first LOT per patient (also supplies age_at_lot_start) ----
idx2 = covars.groupby('mrn')['lot'].idxmin()
patient_covars = covars.loc[idx2].reset_index(drop=True)
assert patient_covars['mrn'].is_unique, 'more than one line-1 row per patient'
print(f'{len(patient_covars):,} patients with covariates')

## Load LLM patient-level cohort and build age groups

`llm_merged` here restricts the cohort to patients who have LLM predictions. Age bins: `<50`, `50-65`, `65-80`, `>80` (right-open).


In [ ]:
llm_patients = pd.read_csv(LLM_PATIENT_PATH, encoding='latin-1', low_memory=False)
llm_patients['mrn'] = llm_patients['mrn'].apply(standardize_mrn)
llm_patients = llm_patients[llm_patients['mrn'].notna()].copy()
for tox in TOXICITY_COLUMNS:
    if tox in llm_patients.columns:
        llm_patients[tox] = pd.to_numeric(llm_patients[tox], errors='coerce').fillna(0).astype(int)
print(f'{len(llm_patients):,} patients with LLM predictions')

llm_merged = llm_patients.merge(patient_covars, on='mrn', how='inner', suffixes=('', '_cov'))
print(f'{len(llm_merged):,} patients after merge (LLM predictions ∩ covariates)')

age_df = llm_merged[['mrn', 'age_at_lot_start']].drop_duplicates('mrn').copy()
age_df['age_at_lot_start'] = pd.to_numeric(age_df['age_at_lot_start'], errors='coerce')
age_df = age_df[age_df['age_at_lot_start'].notna()].copy()
age_df['age_group'] = pd.cut(age_df['age_at_lot_start'], bins=[0, 50, 65, 80, 200],
                              labels=AGE_GROUPS, right=False)
age_df = age_df[age_df['age_group'].notna()].copy()
print(age_df['age_group'].value_counts().reindex(AGE_GROUPS))


## Load batch-level toxicity data and merge with `line1`

Batch-level binary toxicity flags + window dates, restricted to patients with valid line-1
censoring, with `days_from_start` computed relative to `line1_start` and capped at `censor_days`.


In [ ]:
batch_df = pd.read_csv(LLM_BATCH_PATH, encoding='latin-1', low_memory=False)
batch_df['mrn'] = batch_df['mrn'].apply(standardize_mrn)
batch_df = batch_df[batch_df['mrn'].notna()].copy()
batch_df['window_start'] = pd.to_datetime(batch_df['window_start'], errors='coerce')
batch_df['window_end'] = pd.to_datetime(batch_df['window_end'], errors='coerce')
batch_df = batch_df.dropna(subset=['window_start', 'window_end'])
print(f'{len(batch_df):,} batch records for {batch_df["mrn"].nunique():,} patients')

batch_merged = batch_df.merge(line1, on='mrn', how='inner')
batch_merged['days_from_start'] = (batch_merged['window_start'] - batch_merged['line1_start']).dt.days
batch_merged = batch_merged[(batch_merged['days_from_start'] >= 0) &
                            (batch_merged['days_from_start'] <= batch_merged['censor_days'])].copy()
print(f'{len(batch_merged):,} batch records within the censoring window')


## KM 1-year cumulative incidence with 95% CI

For a given set of MRNs and a toxicity, build a
time-to-first-AE survival object (censored at `censor_days`), fit a Kaplan-Meier curve, and
read off the cumulative incidence (1 − survival) and its 95% CI at 12 months.


In [ ]:
def ci_at_t(mrn_set, tox):
    sub_l1 = line1[line1['mrn'].isin(mrn_set)].copy()
    sub_b = batch_merged[batch_merged['mrn'].isin(mrn_set)].copy()
    if len(sub_l1) < 10:
        return 0.0, 0.0, 0.0
    ae_rec = sub_b[sub_b[tox] == 1].copy() if tox in sub_b.columns else pd.DataFrame()
    ae_rec = ae_rec[ae_rec['days_from_start'] >= 0] if len(ae_rec) > 0 else ae_rec
    if len(ae_rec) > 0:
        first_ae = ae_rec.groupby('mrn')['days_from_start'].min().reset_index().rename(
            columns={'days_from_start': 'time'})
        first_ae['event'] = 1
    else:
        first_ae = pd.DataFrame(columns=['mrn', 'time', 'event'])
    surv = sub_l1[['mrn', 'censor_days']].merge(first_ae, on='mrn', how='left')
    surv['event'] = surv['event'].fillna(0).astype(int)
    surv.loc[surv['event'] == 0, 'time'] = surv.loc[surv['event'] == 0, 'censor_days']
    surv.loc[(surv['event'] == 1) & (surv['time'] > surv['censor_days']), 'event'] = 0
    surv.loc[surv['event'] == 0, 'time'] = surv['censor_days']
    surv = surv[surv['time'] > 0].copy()
    if len(surv) < 10:
        return 0.0, 0.0, 0.0
    surv['time_months'] = surv['time'] / 30.44
    kmf = KaplanMeierFitter()
    kmf.fit(surv['time_months'], surv['event'])
    ci = (1 - kmf.survival_function_at_times(T_MONTHS).values[0]) * 100
    ci_tbl = kmf.confidence_interval_survival_function_
    idx = max(0, min(np.searchsorted(kmf.survival_function_.index, T_MONTHS, side='right') - 1,
                     len(ci_tbl) - 1))
    lo = (1 - ci_tbl.iloc[idx, 1]) * 100
    hi = (1 - ci_tbl.iloc[idx, 0]) * 100
    return ci, lo, hi


## Compute CI for every age group × toxicity, and export the values to CSV


In [ ]:
all_mrns = set(line1['mrn'])
records = []
group_mrns = {}

for grp in AGE_GROUPS:
    grp_mrns = set(age_df.loc[age_df['age_group'] == grp, 'mrn']) & all_mrns
    group_mrns[grp] = grp_mrns
    if len(grp_mrns) < 10:
        continue
    for tox in TOXICITY_COLUMNS:
        pct, lo, hi = ci_at_t(grp_mrns, tox)
        records.append({
            'age_group': grp,
            'n_patients': len(grp_mrns),
            'toxicity': tox,
            'toxicity_display': TOXICITY_DISPLAY[tox],
            'cumulative_incidence_pct': round(pct, 3),
            'ci_95_lower': round(lo, 3),
            'ci_95_upper': round(hi, 3),
        })

results_df = pd.DataFrame(records)
results_df.to_csv(CSV_OUT, index=False)
print(f'Saved: {os.path.basename(CSV_OUT)}')
results_df


## Cox PH: fixed reference group (<50) vs. each other age group, per toxicity (Multivariable)

For each toxicity, every other age group is compared against the **same fixed control group,
`<50`** (`AGE_GROUPS[0]`) -- i.e. `<50` vs. `50-65`, `<50` vs. `65-80`, `<50` vs. `>80`. This is
a fixed reference arm (not chosen by which group happens to have the most patients). Each
comparison is a **multivariable** Cox proportional-hazards model matching the covariate adjustment
used in the main figure panels (2C, 2D, 2E).

The model includes:
- `group`: binary (0 = reference age group, 1 = comparator age group) — the exposure of interest
- `has_pd1_flag`, `has_ctla4_flag`: binary treatment flags (combined from raw columns)
- `contains_chemo`, `contains_hormone`, `contains_biologic`, `contains_targeted`: binary treatment flags
- `sex_female`: binary (0 = Male, 1 = Female)
- `cancer_type_*`: dummy variables for cancer type (largest category as reference)

The Wald p-value on `group` is reported as the actual number (not a star/ns bucket); a two-sample
log-rank test (unadjusted) is computed on the same survival data as an independent cross-check.

Requires >=10 patients per arm and >=5 events pooled; otherwise NaN.


In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test

# ============================================================================
# Prepare covariate data for multivariable model
# Matches the adjustment covariates used in main figure panels (2C, 2D, 2E)
# ============================================================================

def _flag_on(x):
    """Convert various flag formats to boolean."""
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)):
        return x == 1
    return str(x).strip().lower() in ('1', 'true', 'yes')

# Build covariate dataframe with all adjustment variables
covar_cols = ['mrn', 'sex', 'cancer_type',
              'contains_ctla4_immuno', 'contains_ctla4', 'contains_non_ctla4_immuno', 'contains_pd1',
              'contains_chemo', 'contains_hormone', 'contains_biologic', 'contains_targeted']
covar_df = patient_covars[[c for c in covar_cols if c in patient_covars.columns]].drop_duplicates('mrn').copy()

# Clean sex
covar_df['sex_clean'] = covar_df['sex'].astype(str).str.strip().str.capitalize()
covar_df = covar_df[covar_df['sex_clean'].isin(['Male', 'Female'])].copy()
covar_df['sex_female'] = (covar_df['sex_clean'] == 'Female').astype(int)

# Combined immuno flags (same logic as panel 2E)
covar_df['has_ctla4_flag'] = (covar_df['contains_ctla4_immuno'].apply(_flag_on) |
                              covar_df['contains_ctla4'].apply(_flag_on)).astype(int)
covar_df['has_pd1_flag'] = (covar_df['contains_non_ctla4_immuno'].apply(_flag_on) |
                            covar_df['contains_pd1'].apply(_flag_on)).astype(int)

# Treatment flags
for col in ['contains_chemo', 'contains_hormone', 'contains_biologic', 'contains_targeted']:
    if col in covar_df.columns:
        covar_df[col] = covar_df[col].apply(_flag_on).astype(int)
    else:
        covar_df[col] = 0

# Clean cancer_type: fill missing with 'Unknown', standardize
covar_df['cancer_type'] = covar_df['cancer_type'].fillna('Unknown').astype(str).str.strip()
cancer_type_counts = covar_df['cancer_type'].value_counts()
CANCER_TYPE_REF = cancer_type_counts.index[0]  # Most common = reference

ADJUST_COLS = ['has_pd1_flag', 'has_ctla4_flag', 'contains_chemo', 'contains_hormone',
               'contains_biologic', 'contains_targeted']
print(f"Adjustment covariates: {ADJUST_COLS} + sex + cancer_type")
print(f"Cancer type reference (most common): {CANCER_TYPE_REF} (n={cancer_type_counts.iloc[0]:,})")
print(f"Cancer types: {len(cancer_type_counts)}")


def _build_surv(mrn_set, tox):
    """Time-to-first-AE survival table -- identical censoring/event logic to `ci_at_t` above,
    centralized here so the KM curve and the Cox model are built from the same survival object."""
    sub_l1 = line1[line1['mrn'].isin(mrn_set)].copy()
    sub_b = batch_merged[batch_merged['mrn'].isin(mrn_set)].copy()
    if len(sub_l1) < 10:
        return None
    ae_rec = sub_b[sub_b[tox] == 1].copy() if tox in sub_b.columns else pd.DataFrame()
    ae_rec = ae_rec[ae_rec['days_from_start'] >= 0] if len(ae_rec) > 0 else ae_rec
    if len(ae_rec) > 0:
        first_ae = ae_rec.groupby('mrn')['days_from_start'].min().reset_index().rename(
            columns={'days_from_start': 'time'})
        first_ae['event'] = 1
    else:
        first_ae = pd.DataFrame(columns=['mrn', 'time', 'event'])
    surv = sub_l1[['mrn', 'censor_days']].merge(first_ae, on='mrn', how='left')
    surv['event'] = surv['event'].fillna(0).astype(int)
    surv.loc[surv['event'] == 0, 'time'] = surv.loc[surv['event'] == 0, 'censor_days']
    surv.loc[(surv['event'] == 1) & (surv['time'] > surv['censor_days']), 'event'] = 0
    surv.loc[surv['event'] == 0, 'time'] = surv['censor_days']
    surv = surv[surv['time'] > 0].copy()
    if len(surv) < 10:
        return None
    surv['time_months'] = surv['time'] / 30.44
    return surv


def cox_hr_pvalue_multivariable(ref_mrns, cmp_mrns, tox):
    """Multivariable Cox PH: comparison age group vs reference age group, adjusting for
    treatment flags, sex (binary), and cancer_type (dummy-encoded). Matches the covariate
    adjustment used in main figure panels."""
    n_ref_input, n_cmp_input = len(ref_mrns), len(cmp_mrns)
    if n_ref_input < 10 or n_cmp_input < 10:
        return np.nan, np.nan, np.nan, np.nan, n_ref_input, n_cmp_input, np.nan, np.nan
    
    surv = _build_surv(set(ref_mrns) | set(cmp_mrns), tox)
    if surv is None:
        return np.nan, np.nan, np.nan, np.nan, n_ref_input, n_cmp_input, np.nan, np.nan
    
    # Add age group indicator: 1 = comparator, 0 = reference
    surv['group'] = surv['mrn'].isin(cmp_mrns).astype(int)
    
    # Merge all covariates
    merge_cols = ['mrn', 'sex_female', 'cancer_type'] + ADJUST_COLS
    surv = surv.merge(covar_df[merge_cols], on='mrn', how='left')
    
    # Drop rows with missing required covariates
    surv = surv.dropna(subset=['sex_female', 'cancer_type'])
    
    if len(surv) < 20:
        return np.nan, np.nan, np.nan, np.nan, n_ref_input, n_cmp_input, np.nan, np.nan
    
    n_ref = int((surv['group'] == 0).sum())
    n_cmp = int((surv['group'] == 1).sum())
    events_ref = int(surv.loc[surv['group'] == 0, 'event'].sum())
    events_cmp = int(surv.loc[surv['group'] == 1, 'event'].sum())
    
    if surv['event'].sum() < 5 or surv['group'].nunique() < 2:
        return np.nan, np.nan, np.nan, np.nan, n_ref, n_cmp, events_ref, events_cmp
    
    # Dummy-encode cancer_type (drop most common as reference)
    cancer_dummies = pd.get_dummies(surv['cancer_type'], prefix='cancer', drop_first=False)
    ref_col = f'cancer_{CANCER_TYPE_REF}'
    if ref_col in cancer_dummies.columns:
        cancer_dummies = cancer_dummies.drop(columns=[ref_col])
    
    # Build model dataframe with all covariates
    # Order: exposure (age group), treatment flags, sex, cancer_type dummies
    model_cols = ['time_months', 'event', 'group'] + ADJUST_COLS + ['sex_female']
    model_df = pd.concat([surv[model_cols].reset_index(drop=True), 
                          cancer_dummies.reset_index(drop=True)], axis=1)
    
    # Fill any NaN in treatment flags with 0
    for col in ADJUST_COLS:
        model_df[col] = model_df[col].fillna(0).astype(int)
    
    # Fit multivariable Cox model with light ridge penalty for stability
    cph = CoxPHFitter(penalizer=0.1, l1_ratio=0.0)
    try:
        cph.fit(model_df, duration_col='time_months', event_col='event')
    except Exception as e:
        print(f"  Cox fit failed for {tox}: {e}")
        return np.nan, np.nan, np.nan, np.nan, n_ref, n_cmp, events_ref, events_cmp
    
    hr = float(np.exp(cph.params_['group']))
    hr_lower = float(cph.summary.loc['group', 'exp(coef) lower 95%'])
    hr_upper = float(cph.summary.loc['group', 'exp(coef) upper 95%'])
    p = float(cph.summary.loc['group', 'p'])
    return hr, hr_lower, hr_upper, p, n_ref, n_cmp, events_ref, events_cmp


def logrank_pvalue(ref_mrns, cmp_mrns, tox):
    """Two-sample log-rank test on the same survival data, as an independent unadjusted cross-check."""
    surv = _build_surv(set(ref_mrns) | set(cmp_mrns), tox)
    if surv is None:
        return np.nan
    is_cmp = surv['mrn'].isin(cmp_mrns)
    if is_cmp.sum() < 10 or (~is_cmp).sum() < 10:
        return np.nan
    try:
        lr = logrank_test(
            surv.loc[is_cmp, 'time_months'], surv.loc[~is_cmp, 'time_months'],
            event_observed_A=surv.loc[is_cmp, 'event'], event_observed_B=surv.loc[~is_cmp, 'event'],
        )
        return float(lr.p_value)
    except Exception:
        return np.nan


# Reference (control) group = the youngest/leftmost age bin, '<50' -- every other group is
# compared against this same fixed reference, not against each other.
ref_group = AGE_GROUPS[0]  # '<50'
cmp_groups = [g for g in AGE_GROUPS if g != ref_group]
print(f"\nReference age group (fixed control, N={len(group_mrns[ref_group]):,}): '{ref_group}'")
for g in cmp_groups:
    print(f"  comparator '{g}': N={len(group_mrns[g]):,}")

cox_records = []
for tox in TOXICITY_COLUMNS:
    print(f"Fitting multivariable Cox for: {tox}")
    for cmp_grp in cmp_groups:
        hr, hr_lo, hr_hi, p, n_ref, n_cmp, ev_ref, ev_cmp = cox_hr_pvalue_multivariable(
            group_mrns[ref_group], group_mrns[cmp_grp], tox
        )
        lr_p = logrank_pvalue(group_mrns[ref_group], group_mrns[cmp_grp], tox)
        cox_records.append({
            'toxicity': tox,
            'toxicity_display': TOXICITY_DISPLAY[tox],
            'reference_group': ref_group,
            'comparison_group': cmp_grp,
            'n_reference': n_ref,
            'n_comparison': n_cmp,
            'events_reference': ev_ref,
            'events_comparison': ev_cmp,
            'hazard_ratio': round(hr, 3) if pd.notna(hr) else np.nan,
            'hr_lower_95': round(hr_lo, 3) if pd.notna(hr_lo) else np.nan,
            'hr_upper_95': round(hr_hi, 3) if pd.notna(hr_hi) else np.nan,
            'p_value_cox': p,
            'p_value_logrank_unadj': lr_p,
        })

cox_df = pd.DataFrame(cox_records)

pmap = cox_df.set_index(['toxicity', 'comparison_group'])[['hazard_ratio', 'hr_lower_95', 'hr_upper_95', 'p_value_cox', 'p_value_logrank_unadj']]
results_df = results_df.merge(
    pmap, left_on=['toxicity', 'age_group'], right_index=True, how='left'
)
results_df = results_df.rename(columns={
    'hazard_ratio': 'hr_vs_reference_adj',
    'hr_lower_95': 'hr_vs_reference_adj_lower_95',
    'hr_upper_95': 'hr_vs_reference_adj_upper_95',
    'p_value_cox': 'p_value_vs_reference_adj',
    'p_value_logrank_unadj': 'p_value_logrank_vs_reference_unadj',
})
results_df['reference_group'] = ref_group

results_df.to_csv(CSV_OUT, index=False)
print(f'\nSaved (with multivariable Cox + unadjusted log-rank p-values): {os.path.basename(CSV_OUT)}')
print(f'Adjustments: sex, cancer_type (reference: {CANCER_TYPE_REF})')
cox_df

## What the p-value is adjusted for

**Covariate-adjusted (multivariable).** `p_value_vs_reference_adj` is from a **multivariable** Cox model
that includes the same adjustment covariates used in the main figure panels (2C, 2D, 2E):

- Treatment: `has_pd1_flag`, `has_ctla4_flag`, `contains_chemo`, `contains_hormone`, `contains_biologic`, `contains_targeted`
- Demographics: sex (binary), cancer type (dummy-encoded categorical)

The HR for the age group comparison represents the comparator vs. reference hazard ratio **after
adjusting for** confounding by treatment regimen, sex, and cancer type.

**Not adjusted for multiple comparisons, by default.** Six toxicities x three comparator age
groups = 18 tests here, each tested independently, which inflates the family-wise false-positive
rate. The cell below adds Benjamini-Hochberg FDR-adjusted and Bonferroni-adjusted p-values as
extra columns (`p_value_vs_reference_adj_fdr_bh`, `p_value_vs_reference_adj_bonferroni`) -- both land in
the CSV. The figure itself still displays the **covariate-adjusted** Cox p-value on the brackets by
default -- switch `PLOT_ADJUSTED_P` in the plot cell to `'fdr_bh'` or `'bonferroni'` if the
multiplicity-adjusted value should be shown instead.


In [ ]:
def bh_fdr(pvals):
    """Benjamini-Hochberg FDR-adjusted p-values (q-values), NaN-safe."""
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    valid_idx = np.where(~np.isnan(p))[0]
    m = len(valid_idx)
    if m == 0:
        return out
    vp = p[valid_idx]
    order = np.argsort(vp)
    ranked_p = vp[order]
    ranks = np.arange(1, m + 1)
    q = ranked_p * m / ranks
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    out[valid_idx[order]] = q
    return out


def bonferroni_adjust(pvals):
    """Bonferroni-adjusted p-values, NaN-safe."""
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    valid_idx = np.where(~np.isnan(p))[0]
    m = len(valid_idx)
    if m == 0:
        return out
    out[valid_idx] = np.clip(p[valid_idx] * m, 0, 1)
    return out


# Family = all toxicity x comparator-group tests together (18 here).
raw_p = cox_df['p_value_cox'].values
cox_df['p_value_fdr_bh'] = bh_fdr(raw_p)
cox_df['p_value_bonferroni'] = bonferroni_adjust(raw_p)

adj_map = cox_df.set_index(['toxicity', 'comparison_group'])[['p_value_fdr_bh', 'p_value_bonferroni']]
results_df = results_df.merge(adj_map, left_on=['toxicity', 'age_group'], right_index=True, how='left')
results_df = results_df.rename(columns={
    'p_value_fdr_bh': 'p_value_vs_reference_adj_fdr_bh',
    'p_value_bonferroni': 'p_value_vs_reference_adj_bonferroni',
})

# ---- document, in the CSV itself, exactly what each p-value column is (and isn't) adjusted for ----
results_df['cumulative_incidence_method'] = (
    'Kaplan-Meier, fit separately within each age group (unadjusted for covariates)'
)
results_df['p_value_vs_reference_covariate_adjustment'] = (
    f'Multivariable Cox PH model adjusted for sex (binary) and cancer_type (categorical, '
    f'reference: {CANCER_TYPE_REF})'
)
results_df['p_value_vs_reference_multiplicity_adjustment'] = (
    'None (raw p-value); see p_value_vs_reference_adj_fdr_bh / p_value_vs_reference_adj_bonferroni for '
    'correction across all toxicity x comparator-group tests'
)
results_df['multiplicity_adjustment_family'] = (
    f'{len(TOXICITY_COLUMNS)} toxicities x {len(cmp_groups)} comparator groups '
    f'= {len(TOXICITY_COLUMNS) * len(cmp_groups)} tests, corrected together'
)

results_df.to_csv(CSV_OUT, index=False)
print(f'Saved (with FDR/Bonferroni-adjusted p-values): {os.path.basename(CSV_OUT)}')
cox_df[['toxicity_display', 'comparison_group', 'p_value_cox', 'p_value_fdr_bh', 'p_value_bonferroni']]

## Plot

Retuned to the Nature 7pt / 6pt / 5pt text tiers
(originally 10pt/9pt/8pt/7pt in the script). No `bbox_inches='tight'` on save — place at 100%
in Illustrator.


In [ ]:
def set_axes_position_inches(fig, ax, left_in, top_in, width_in, height_in):
    fw, fh = fig.get_size_inches()
    ax.set_position([
        left_in / fw,
        1 - (top_in + height_in) / fh,
        width_in / fw,
        height_in / fh,
    ])

FIG_WIDTH_IN  = 3.6
FIG_HEIGHT_IN = 2.3

In [ ]:
PLOT_ADJUSTED_P = 'cox'  # 'cox' (raw), 'fdr_bh', or 'bonferroni'
P_COL = {'cox': 'p_value_cox', 'fdr_bh': 'p_value_fdr_bh', 'bonferroni': 'p_value_bonferroni'}[PLOT_ADJUSTED_P]

def format_pval(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return 'p < 0.001'
    return f'p = {p:.3f}'

ae_list = TOXICITY_COLUMNS
display_names = [TOXICITY_DISPLAY[t] for t in ae_list]
groups = AGE_GROUPS
colors = AGE_GROUP_COLORS
n_ae = len(ae_list)
n_levels = len(cmp_groups)  # stacked p-value brackets per toxicity

fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))
total_width = 0.80
bar_width = total_width / len(groups)
x = np.arange(n_ae)

bar_x_pos = {}
for j, grp in enumerate(groups):
    grp_mrns = group_mrns[grp]
    bar_off = -total_width / 2 + (j + 0.5) * bar_width
    bar_x_pos[grp] = bar_off
    if len(grp_mrns) < 10:
        continue
    sub = results_df[results_df['age_group'] == grp].set_index('toxicity').loc[ae_list]
    g_pct = sub['cumulative_incidence_pct'].to_numpy()
    g_lo = sub['ci_95_lower'].to_numpy()
    g_hi = sub['ci_95_upper'].to_numpy()
    ax.bar(x + bar_off, g_pct, bar_width, yerr=[g_pct - g_lo, g_hi - g_pct], capsize=1.5,
           color=colors[grp], edgecolor='none', label=f'{grp} (N={len(grp_mrns):,})',
           ecolor='gray', error_kw={'linewidth': 0.5})

ax.set_xlabel('Adverse event', fontsize=7)
ax.set_ylabel('1-year cumulative\nincidence (%, 95% CI)', fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels(display_names, rotation=30, ha='right', fontsize=6)
ax.tick_params(axis='y', labelsize=6)
ax.legend(fontsize=5, framealpha=0.95, ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.15),
          handlelength=1, columnspacing=0.8)

# ---- reserve extra headroom above the bars for stacked p-value brackets ----
data_max = ax.get_ylim()[1]
pad_frac = 0.15 + 0.14 * n_levels
new_top = min(data_max * (1 + pad_frac), 60)
ax.set_ylim(0, new_top)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ---- p-value brackets: reference age group vs. each comparator group, per toxicity ----
p_lookup = cox_df.set_index(['toxicity', 'comparison_group'])[P_COL]
y_range = ax.get_ylim()[1]
tick_h = y_range * 0.02
level_gap = y_range * 0.11

for i, tox in enumerate(ae_list):
    tops_all = results_df.loc[results_df['toxicity'] == tox, 'ci_95_upper']
    base_top = tops_all.max() if len(tops_all) else 0
    for lvl, cmp_grp in enumerate(cmp_groups):
        if (tox, cmp_grp) not in p_lookup.index:
            continue
        label = format_pval(p_lookup.loc[(tox, cmp_grp)])
        if not label:
            continue
        x_left = x[i] + bar_x_pos[ref_group]
        x_right = x[i] + bar_x_pos[cmp_grp]
        if x_left > x_right:
            x_left, x_right = x_right, x_left
        y_bar = base_top + level_gap * (lvl + 1)
        y_tick = y_bar - tick_h
        ax.plot([x_left, x_left, x_right, x_right], [y_tick, y_bar, y_bar, y_tick],
                color='black', linewidth=0.5)
        ax.text((x_left + x_right) / 2, y_bar + tick_h * 0.4, label,
                ha='center', va='bottom', fontsize=4.3)

# ---- place with a rough first guess, then measure real overflow and correct ----
# (no bbox_inches='tight' at save time -- figure is placed in Illustrator at 100% scale)
guess_left, guess_bottom, guess_top = 0.5, 0.55, 0.35
set_axes_position_inches(fig, ax, left_in=guess_left, top_in=guess_top,
                          width_in=FIG_WIDTH_IN - guess_left - 0.05,
                          height_in=FIG_HEIGHT_IN - guess_top - guess_bottom)

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
tight_bbox = fig.get_tightbbox(renderer)

overflow_left   = max(0, -tight_bbox.x0)
overflow_bottom = max(0, -tight_bbox.y0)
overflow_right  = max(0, tight_bbox.x1 - FIG_WIDTH_IN)
overflow_top    = max(0, tight_bbox.y1 - FIG_HEIGHT_IN)

LEFT_MARGIN_IN   = guess_left   + overflow_left   + 0.03
BOTTOM_MARGIN_IN = guess_bottom + overflow_bottom + 0.03
RIGHT_MARGIN_IN  = 0.05         + overflow_right  + 0.03
TOP_MARGIN_IN    = guess_top    + overflow_top    + 0.03
PLOT_WIDTH_IN    = FIG_WIDTH_IN  - LEFT_MARGIN_IN - RIGHT_MARGIN_IN
PLOT_HEIGHT_IN   = FIG_HEIGHT_IN - TOP_MARGIN_IN  - BOTTOM_MARGIN_IN

set_axes_position_inches(fig, ax, left_in=LEFT_MARGIN_IN, top_in=TOP_MARGIN_IN,
                          width_in=PLOT_WIDTH_IN, height_in=PLOT_HEIGHT_IN)

print(f"margins (in): left={LEFT_MARGIN_IN:.2f} right={RIGHT_MARGIN_IN:.2f} "
      f"top={TOP_MARGIN_IN:.2f} bottom={BOTTOM_MARGIN_IN:.2f}")
print(f"plot area (in): {PLOT_WIDTH_IN:.2f} x {PLOT_HEIGHT_IN:.2f}")

In [ ]:
with PdfPages(PDF_OUT) as pdf:
    pdf.savefig(fig, dpi=450)
plt.close(fig)
print(f'Saved: {os.path.basename(PDF_OUT)}')